***Setup and imports***

In [1]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = os.getenv("GROQ_MODEL")
print(f"Client ready ✅ — using {MODEL}")

Client ready ✅ — using llama-3.1-8b-instant


***Load a document***

In [9]:
document = """
Retrieval-Augmented Generation (RAG) is a technique that combines 
information retrieval with text generation. Instead of relying solely 
on a language model's parametric knowledge, RAG retrieves relevant 
documents from an external knowledge base and uses them as context 
for generating responses.

RAG has two main components:
1. Retriever: searches a document store for relevant chunks
2. Generator: an LLM that uses retrieved chunks to answer questions

RAG is preferred over fine-tuning for factual accuracy because 
knowledge in the retrieval store is always current and citeable, 
whereas fine-tuned knowledge can go stale and hallucinate.
"""

print(f"Document loaded ✅ — {len(document.split())} words")

Document loaded ✅ — 92 words


***Q&A Function***

In [3]:
def ask(question, doc):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """You are a precise document analyst. 
Answer questions strictly based on the provided document. 
If the answer is not in the document, say 'Not found in document.'"""},
            {"role": "user", "content": f"Document:\n{doc}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content

# Test it
print(ask("What are the two main components of RAG?", document))

The two main components of RAG are:

1. Retriever: searches a document store for relevant chunks
2. Generator: an LLM (Large Language Model) that uses retrieved chunks to answer questions


***Ask multiple questions***

In [4]:
questions = [
    "Why is RAG preferred over fine-tuning?",
    "What does the retriever do?",
    "What is the capital of France?"  # not in document — tests boundary
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q, document)}")
    print("-" * 50)

Q: Why is RAG preferred over fine-tuning?
A: RAG is preferred over fine-tuning because knowledge in the retrieval store is always current and citeable, whereas fine-tuned knowledge can go stale and hallucinate.
--------------------------------------------------
Q: What does the retriever do?
A: The retriever searches a document store for relevant chunks.
--------------------------------------------------
Q: What is the capital of France?
A: Not found in document.
--------------------------------------------------


***Load from a real text file***

In [5]:
# Create a sample file first
sample_text = """
LangChain is a framework for building applications powered by large language models.
It provides tools for chaining LLM calls, connecting to external data sources,
and building agents that can use tools to complete tasks.

Key components of LangChain:
1. Chains: sequences of LLM calls linked together
2. Agents: LLMs that decide which tools to use and when
3. Memory: storing and retrieving conversation history
4. Tools: external functions an agent can call (search, calculator, APIs)

LangChain supports multiple LLM providers including OpenAI, Anthropic, and Groq.
"""

# Write to file
with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

# Read it back
with open("sample_doc.txt", "r") as f:
    loaded_doc = f.read()

print(f"File loaded ✅ — {len(loaded_doc.split())} words")
print(ask("What are the key components of LangChain?", loaded_doc))

File loaded ✅ — 85 words
The key components of LangChain are:

1. Chains: sequences of LLM calls linked together
2. Agents: LLMs that decide which tools to use and when
3. Memory: storing and retrieving conversation history
4. Tools: external functions an agent can call (search, calculator, APIs)


***Track cost per question***

In [6]:
def ask_with_stats(question, doc):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer questions strictly based on the provided document."},
            {"role": "user", "content": f"Document:\n{doc}\n\nQuestion: {question}"}
        ]
    )
    answer = response.choices[0].message.content
    tokens = response.usage.total_tokens
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"Tokens used: {tokens}")
    print("-" * 50)

ask_with_stats("What tools can a LangChain agent use?", loaded_doc)
ask_with_stats("Who invented LangChain?", loaded_doc)  # not in doc

Q: What tools can a LangChain agent use?
A: Based on the provided document, a LangChain agent can use various external functions, referred to as "Tools". These tools can include:

1. Search
2. Calculator
3. APIs (Applications Programming Interfaces) 

These tools are designed to be used by the agent to complete tasks, and the specifics of each tool can be further developed or customized.
Tokens used: 248
--------------------------------------------------
Q: Who invented LangChain?
A: The document does not mention who invented LangChain. It only provides information about the framework's purpose, key components, and supported LLM providers, but does not mention any developers or founders.
Tokens used: 211
--------------------------------------------------
